# Qwen3-Reranker-0.6B：Colab T4 smoke benchmark

本 notebook 用项目 source ZIP（默认）或可选的公开仓库 clone，以及公开的 `Qwen/Qwen3-Reranker-0.6B` checkpoint 做一个可复现的 5-sample smoke benchmark。运行前在 Colab 选择 `Runtime → Change runtime type → T4 GPU`。

流程：

1. 默认上传本地生成的 source ZIP（不需要 GitHub 登录）；也可以把 `SOURCE_MODE` 改为 `clone` 尝试公开仓库；
2. 将 source ZIP 安全解压到 `/content/techjam/repo`，并校验 benchmark、public set 与 CUDA 能力标记；
3. 安装与实验环境一致的 pinned 依赖；
4. 通过 Colab upload **只上传** `catalog.jsonl`（`public_set.jsonl` 从 source ZIP/仓库读取）；
5. 用 seed 17 生成 target-free、scenario-stratified manifest；
6. 在固定 Hugging Face revision `e61197ed45024b0ed8a2d74b80b4d909f1255473` 下载 checkpoint；
7. 对同一批 5 个 validation samples 跑 feature-only baseline 和 CUDA Top-30 rerank；
8. 下载 manifest、baseline、reranked 和 comparison JSON。

Notebook 不需要 Hugging Face token 或任何 API key。模型下载完成后，benchmark 会强制 offline loading。

In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import shutil
import stat
import subprocess
import sys
import tempfile
import zipfile

REPO_URL = "https://github.com/ByteSize2026/techjam-conversational-search.git"
# Upload is the default because the repository may be private or unavailable
# from Colab. Change to "clone" only when the public GitHub repository is reachable.
SOURCE_MODE = "upload"
COLAB_ROOT = Path("/content/techjam")
REPO_DIR = COLAB_ROOT / "repo"
CACHE_ROOT = COLAB_ROOT / "caches"
for cache_path in (
    COLAB_ROOT,
    CACHE_ROOT / "pip",
    CACHE_ROOT / "tmp",
    CACHE_ROOT / "torch",
    COLAB_ROOT / "models" / "huggingface",
):
    cache_path.mkdir(parents=True, exist_ok=True)
os.environ.update({
    "HF_HOME": str(COLAB_ROOT / "models" / "huggingface"),
    "HUGGINGFACE_HUB_CACHE": str(COLAB_ROOT / "models" / "huggingface"),
    "TRANSFORMERS_CACHE": str(COLAB_ROOT / "models" / "huggingface"),
    "PIP_CACHE_DIR": str(CACHE_ROOT / "pip"),
    "TMPDIR": str(CACHE_ROOT / "tmp"),
    "XDG_CACHE_HOME": str(CACHE_ROOT),
    "TORCH_HOME": str(CACHE_ROOT / "torch"),
})

_REQUIRED_FILES = (
    Path("scripts/benchmark_qwen_reranker.py"),
    Path("data/public_set.jsonl"),
)
_FORBIDDEN_PARTS = {".git", ".claude", ".cursor"}
_FORBIDDEN_MODEL_DIRS = {"models", "model_assets", "model-assets", "checkpoints", "weights"}
_FORBIDDEN_WEIGHT_SUFFIXES = {
    ".bin", ".ckpt", ".gguf", ".onnx", ".pt", ".pth",
    ".safetensors", ".tflite", ".weights",
}

def _normalise_zip_name(name):
    """Return a safe POSIX member name, rejecting traversal and absolute paths."""
    if not isinstance(name, str) or not name or "\x00" in name:
        raise RuntimeError("Source ZIP contains an invalid member name")
    normalised = name.replace("\\", "/")
    if normalised.startswith("/") or PurePosixPath(normalised).is_absolute():
        raise RuntimeError(f"Source ZIP contains an absolute path: {name!r}")
    raw_parts = normalised.split("/")
    if any(part == ".." for part in raw_parts):
        raise RuntimeError(f"Source ZIP contains a traversal path: {name!r}")
    parts = tuple(part for part in raw_parts if part not in ("", "."))
    if not parts:
        return ""
    return "/".join(parts)

def _reject_forbidden_zip_member(member_name):
    parts = member_name.split("/")
    lower_parts = [part.lower() for part in parts]
    basename = lower_parts[-1]
    if any(part in _FORBIDDEN_PARTS for part in lower_parts):
        return "repository metadata or credentials"
    if any(part == ".env" or part.startswith(".env.") for part in lower_parts):
        return "environment/secret file"
    if basename in {"catalog.jsonl", "catalog.jsonl.gz"}:
        return "catalog asset"
    if any(part in _FORBIDDEN_MODEL_DIRS for part in lower_parts):
        return "model asset directory"
    if any(basename.endswith(suffix) for suffix in _FORBIDDEN_WEIGHT_SUFFIXES):
        return "model weight file"
    if basename.startswith("pytorch_model") or basename in {"model.safetensors.index.json", "weights.index.json"}:
        return "model weight index"
    return None

def _safe_extract_source_zip(zip_bytes, archive_name):
    """Validate and extract an uploaded source archive without Zip Slip."""
    with tempfile.TemporaryDirectory(dir=COLAB_ROOT) as staging_name:
        staging_dir = Path(staging_name) / "extracted"
        staging_dir.mkdir()
        archive_path = Path(staging_name) / "source.zip"
        archive_path.write_bytes(zip_bytes)
        try:
            archive = zipfile.ZipFile(archive_path)
        except (OSError, zipfile.BadZipFile) as exc:
            raise RuntimeError(f"Uploaded source is not a readable ZIP: {archive_name}") from exc
        with archive:
            seen = set()
            members = []
            for info in archive.infolist():
                member_name = _normalise_zip_name(info.filename)
                if not member_name:
                    continue
                if member_name in seen:
                    raise RuntimeError(f"Source ZIP contains duplicate member: {member_name}")
                seen.add(member_name)
                reason = _reject_forbidden_zip_member(member_name)
                if reason:
                    raise RuntimeError(f"Source ZIP must not contain {reason}: {member_name}")
                mode = (info.external_attr >> 16) & 0o170000
                if mode == stat.S_IFLNK:
                    raise RuntimeError(f"Source ZIP contains a symbolic link: {member_name}")
                members.append((info, member_name))
            for info, member_name in members:
                destination = staging_dir / member_name
                destination.parent.mkdir(parents=True, exist_ok=True)
                if info.is_dir():
                    destination.mkdir(parents=True, exist_ok=True)
                    continue
                with archive.open(info) as source, destination.open("wb") as target:
                    shutil.copyfileobj(source, target)
        source_root = _repo_source_root(staging_dir)
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        shutil.copytree(source_root, REPO_DIR, symlinks=False)

def _repo_source_root(extracted_dir):
    """Support a direct archive and a single top-level-folder archive."""
    if (extracted_dir / _REQUIRED_FILES[0]).is_file() and (extracted_dir / _REQUIRED_FILES[1]).is_file():
        return extracted_dir
    children = [child for child in extracted_dir.iterdir() if child.name not in {"__MACOSX"}]
    if len(children) == 1 and children[0].is_dir():
        return children[0]
    return extracted_dir

def _validate_repo(repo_dir, source_label):
    missing = [str(path) for path in _REQUIRED_FILES if not (repo_dir / path).is_file()]
    if not (repo_dir / "starter").is_dir():
        missing.append("starter/")
    if missing:
        raise RuntimeError(f"Source {source_label} is missing required paths: {', '.join(missing)}")
    benchmark_script = repo_dir / _REQUIRED_FILES[0]
    benchmark_source = benchmark_script.read_text(encoding="utf-8")
    required_markers = ("frozen_trace_path", "qwen_reranker_device", "cuda")
    missing_markers = [marker for marker in required_markers if marker not in benchmark_source]
    help_result = subprocess.run(
        [sys.executable, str(benchmark_script), "rerank", "--help"],
        cwd=repo_dir,
        text=True,
        capture_output=True,
    )
    help_output = (help_result.stdout or "") + (help_result.stderr or "")
    if help_result.returncode or "--device" not in help_output or "cuda" not in help_output.lower():
        missing_markers.append("rerank --help (--device cuda)")
    if missing_markers:
        raise RuntimeError(
            f"Source {source_label} does not contain the required CUDA benchmark capability; "
            "use the current source ZIP (or push the CUDA patch before using clone). Missing: "
            + ", ".join(missing_markers)
        )
    return benchmark_script

if SOURCE_MODE not in {"upload", "clone"}:
    raise ValueError("SOURCE_MODE must be either 'upload' or 'clone'")

source_label = SOURCE_MODE
source_digest = None
if SOURCE_MODE == "upload":
    from google.colab import files

    uploaded_source = files.upload()
    uploaded_names = sorted(uploaded_source)
    if len(uploaded_names) != 1 or not uploaded_names[0].lower().endswith(".zip"):
        raise ValueError("Upload exactly one source ZIP containing the project code; do not upload catalog or model files.")
    source_name = uploaded_names[0]
    source_bytes = uploaded_source[source_name]
    if not source_bytes:
        raise ValueError("The uploaded source ZIP is empty")
    source_digest = hashlib.sha256(source_bytes).hexdigest()
    _safe_extract_source_zip(source_bytes, source_name)
    source_label = f"uploaded ZIP {source_name}"
elif SOURCE_MODE == "clone":
    if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
        shutil.rmtree(REPO_DIR)
    if not (REPO_DIR / ".git").is_dir():
        clone_result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            text=True,
            capture_output=True,
        )
        if clone_result.returncode:
            details = (clone_result.stderr or clone_result.stdout or "no git output").strip()
            raise RuntimeError(
                "Public repository clone failed (HTTP 404 usually means the repository is private or unavailable). "
                "Set SOURCE_MODE = 'upload' and upload the project source ZIP; no GitHub PAT/token is required.\n"
                + details[-2000:]
            )
    else:
        remote_result = subprocess.run(
            ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"],
            text=True,
            capture_output=True,
        )
        remote = (remote_result.stdout or "").strip()
        if remote_result.returncode or remote != REPO_URL:
            raise RuntimeError(f"Existing checkout has unexpected origin: {remote or '<unknown>'}")
    source_label = f"public clone {REPO_URL}"

BENCHMARK_SCRIPT = _validate_repo(REPO_DIR, source_label)
os.chdir(REPO_DIR)
commit = (
    subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    if (REPO_DIR / ".git").is_dir()
    else f"uploaded-source-sha256:{source_digest}"
)
print(f"Source:    {source_label}")
print(f"Checkout:  {REPO_DIR}")
print(f"Revision:  {commit}")
print("CUDA benchmark capability check: OK")

## Install pinned runtime

These packages match the validated experiment environment. Colab's pre-installed CUDA-enabled `torch` is deliberately preserved; `--no-cache-dir` keeps pip's cache from consuming additional disk space.

In [ ]:
PINNED_PACKAGES = [
    "transformers==5.15.1",
    "sentence-transformers==6.0.0",
    "huggingface-hub==1.28.0",
    "safetensors==0.8.0",
    "tokenizers==0.22.2",
    "jedi==0.19.2",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--no-cache-dir", *PINNED_PACKAGES],
    check=True,
)

subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
print("pip check: OK")
print("Pinned packages:")
for package in PINNED_PACKAGES:
    print("  " + package)

## T4 / CUDA guard

The benchmark is intentionally CUDA-only. A non-T4 NVIDIA GPU is allowed for experimentation, but the reported resource profile is then not the requested T4 profile.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. In Colab choose Runtime → Change runtime type → T4 GPU, then restart."
    )
gpu_name = torch.cuda.get_device_name(0)
cuda_version = torch.version.cuda
print(f"GPU: {gpu_name}")
print(f"CUDA runtime: {cuda_version}")
if "t4" not in gpu_name.lower():
    print("WARNING: this is not a Tesla T4; continue only if a different CUDA GPU is intentional.")
else:
    print("T4 guard: OK")
torch.cuda.empty_cache()

## Upload the catalog (only this file)

Do not upload `public_set.jsonl`, model files, credentials, or any other file. The public session file is cloned from the repository; the evaluator is the only component that reads its labels.

In [ ]:
from google.colab import files

uploaded = files.upload()
uploaded_names = sorted(uploaded)
if uploaded_names != ["catalog.jsonl"]:
    raise ValueError(
        f"Upload exactly one file named catalog.jsonl; received {uploaded_names!r}"
    )
catalog_path = REPO_DIR / "data" / "catalog.jsonl"
catalog_bytes = uploaded["catalog.jsonl"]
if not catalog_bytes:
    raise ValueError("catalog.jsonl is empty")
catalog_path.write_bytes(catalog_bytes)
print(f"Wrote {catalog_path} ({catalog_path.stat().st_size:,} bytes)")

## Fixed model snapshot

The revision is an immutable commit hash. `snapshot_download` is called without a token because this is a public checkpoint.

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen3-Reranker-0.6B"
MODEL_REVISION = "e61197ed45024b0ed8a2d74b80b4d909f1255473"
MODEL_DIR = COLAB_ROOT / "models" / "qwen3-reranker-0.6b-e61197ed"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        revision=MODEL_REVISION,
        local_dir=str(MODEL_DIR),
    )
)
if not (MODEL_PATH / "config.json").is_file():
    raise RuntimeError(f"Checkpoint snapshot is incomplete: {MODEL_PATH}")
print(f"Model:    {MODEL_ID}")
print(f"Revision: {MODEL_REVISION}")
print(f"Path:     {MODEL_PATH}")

# No model/network fetch is allowed after this point.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

## Seed-17 manifest

The manifest is generated before either benchmark run and is shared by baseline and reranking. Only `sample_id`, scenario, and difficulty metadata are copied into it.

In [ ]:
RESULT_DIR = COLAB_ROOT / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = RESULT_DIR / "qwen3-reranker-manifest-seed17.json"
subprocess.run(
    [
        sys.executable,
        "scripts/benchmark_qwen_reranker.py",
        "manifest",
        "--public-set",
        str(REPO_DIR / "data" / "public_set.jsonl"),
        "--output",
        str(MANIFEST_PATH),
        "--seed",
        "17",
    ],
    cwd=REPO_DIR,
    check=True,
)
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
if manifest.get("seed") != 17:
    raise RuntimeError(f"Unexpected manifest seed: {manifest.get('seed')!r}")
print(f"Manifest: {MANIFEST_PATH}")
print({name: len(values) for name, values in manifest["split_ids"].items()})

## 5-sample feature-only baseline

This uses the first five IDs of the seed-17 validation split. No model backend is enabled.

In [ ]:
BASELINE_RESULT = RESULT_DIR / "feature-only-baseline-smoke5.json"
subprocess.run(
    [
        sys.executable,
        "scripts/benchmark_qwen_reranker.py",
        "baseline",
        "--catalog",
        str(catalog_path),
        "--public-set",
        str(REPO_DIR / "data" / "public_set.jsonl"),
        "--manifest",
        str(MANIFEST_PATH),
        "--split",
        "validation",
        "--sample-limit",
        "5",
        "--output",
        str(BASELINE_RESULT),
    ],
    cwd=REPO_DIR,
    check=True,
)
baseline = json.loads(BASELINE_RESULT.read_text(encoding="utf-8"))
print(json.dumps({key: baseline.get(key) for key in ("sample_count", "hit_rate_at_10", "mrr", "mttc", "recommended_technical_score")}, indent=2))

## 5-sample Top-30 Qwen rerank

The deterministic agent supplies at most 30 catalog-valid candidates; Qwen only reorders that whitelist. `--device cuda` is required by the guard above. A 60-second soft timeout keeps slow batches from stalling the smoke run; a timeout/failure preserves the feature-only order.

In [ ]:
RERANKED_RESULT = RESULT_DIR / "qwen3-reranked-top30-smoke5.json"
subprocess.run(
    [
        sys.executable,
        "scripts/benchmark_qwen_reranker.py",
        "rerank",
        "--catalog",
        str(catalog_path),
        "--public-set",
        str(REPO_DIR / "data" / "public_set.jsonl"),
        "--manifest",
        str(MANIFEST_PATH),
        "--split",
        "validation",
        "--sample-limit",
        "5",
        "--model-path",
        str(MODEL_PATH),
        "--revision",
        MODEL_REVISION,
        "--device",
        "cuda",
        "--batch-size",
        "8",
        "--candidate-limit",
        "30",
        "--timeout-seconds",
        "60",
        "--fusion-weight",
        "1.0",
        "--output",
        str(RERANKED_RESULT),
    ],
    cwd=REPO_DIR,
    check=True,
)
reranked = json.loads(RERANKED_RESULT.read_text(encoding="utf-8"))
print(json.dumps({key: reranked.get(key) for key in ("sample_count", "hit_rate_at_10", "mrr", "mttc", "recommended_technical_score")}, indent=2))

## Compare and download results

In [ ]:
COMPARISON_RESULT = RESULT_DIR / "qwen3-reranker-comparison-smoke5.json"
subprocess.run(
    [
        sys.executable,
        "scripts/benchmark_qwen_reranker.py",
        "compare",
        "--baseline",
        str(BASELINE_RESULT),
        "--reranked",
        str(RERANKED_RESULT),
        "--manifest",
        str(MANIFEST_PATH),
        "--split",
        "validation",
        "--sample-limit",
        "5",
        "--output",
        str(COMPARISON_RESULT),
    ],
    cwd=REPO_DIR,
    check=True,
)
comparison = json.loads(COMPARISON_RESULT.read_text(encoding="utf-8"))
print(json.dumps(comparison["comparisons"][0]["delta"], indent=2))

RESULT_BUNDLE = COLAB_ROOT / "qwen3-reranker-smoke5-results.zip"
import zipfile
with zipfile.ZipFile(RESULT_BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (MANIFEST_PATH, BASELINE_RESULT, RERANKED_RESULT, COMPARISON_RESULT):
        archive.write(path, arcname=path.name)
print(f"Result bundle: {RESULT_BUNDLE}")
files.download(str(RESULT_BUNDLE))